# Simple RAG (Retrieval-Augmented Generation) System

This notebook builds a beginner-friendly **RAG pipeline** that answers questions from your own documents (PDFs, `.txt`, notes, resumes, research papers, etc.).

**Pipeline overview:**
1. Load documents (PDF/TXT) from a folder
2. Split documents into overlapping chunks
3. Embed chunks using a sentence-transformer model
4. Store embeddings in a FAISS vector index (retrieval)
5. Given a question, retrieve the most relevant chunks
6. Feed the question + retrieved chunks into a language model to generate an answer (generation)

Everything here runs locally / free using open-source models (`sentence-transformers` + `flan-t5`), so no API key is required. If you have an OpenAI/Anthropic key, a swap-in cell is provided at the end to use a stronger LLM instead.


## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers pypdf accelerate


## 2. Imports

In [ ]:
import os
import glob
import textwrap
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline


## 3. Load your documents

Put your `.pdf` or `.txt` files inside a folder called `docs/` (create it if it doesn't exist), or just edit `DOCS_FOLDER` below to point to your own folder.

For this demo we also auto-create a small sample `.txt` file so the notebook runs end-to-end even if you haven't added your own documents yet.


In [ ]:
DOCS_FOLDER = "docs"
os.makedirs(DOCS_FOLDER, exist_ok=True)

# Create a small sample document if the folder is empty, just so the notebook
# is runnable out-of-the-box. Delete this cell / file once you add your own docs.
sample_path = os.path.join(DOCS_FOLDER, "sample_notes.txt")
if not os.listdir(DOCS_FOLDER):
    sample_text = '''Retrieval-Augmented Generation (RAG) Notes

RAG is a technique that combines information retrieval with text generation.
Instead of relying only on what a language model learned during training,
RAG first retrieves relevant chunks of text from an external knowledge base
(such as your own PDFs or notes) and then passes those chunks to the language
model as context, so it can generate a more accurate, grounded answer.

Key components of a RAG system:
1. Document loader - reads raw files (PDF, TXT, DOCX, etc.)
2. Text splitter - breaks documents into smaller overlapping chunks
3. Embedding model - converts text chunks into numerical vectors
4. Vector store - stores embeddings and allows fast similarity search (e.g. FAISS)
5. Retriever - given a query, finds the top-k most similar chunks
6. Generator (LLM) - given the query and retrieved chunks, produces a final answer

RAG is especially useful for private or custom data because the language model
was never trained on it, but it can still answer questions about it accurately
as long as the relevant text is retrieved and shown to it as context.
'''
    with open(sample_path, "w") as f:
        f.write(sample_text)
    print(f"No documents found, created a sample file at: {sample_path}")

print("Documents folder contents:", os.listdir(DOCS_FOLDER))


## 4. Extract raw text from all documents (PDF + TXT)

In [ ]:
def load_text_from_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += page_text + "\n"
    return text

def load_text_from_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_all_documents(folder):
    documents = []  # list of (filename, full_text)
    for path in glob.glob(os.path.join(folder, "*")):
        if path.lower().endswith(".pdf"):
            text = load_text_from_pdf(path)
        elif path.lower().endswith(".txt"):
            text = load_text_from_txt(path)
        else:
            continue
        documents.append((os.path.basename(path), text))
    return documents

documents = load_all_documents(DOCS_FOLDER)
print(f"Loaded {len(documents)} document(s):")
for name, text in documents:
    print(f" - {name}: {len(text)} characters")


## 5. Split documents into chunks

Language models and embedding models work best on short passages, so we split each document
into overlapping chunks (default: 500 characters with 50 characters of overlap).


In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    text_length = len(text)
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Build a flat list of chunks, each tagged with its source document
all_chunks = []       # the actual text chunks
chunk_sources = []    # which file each chunk came from

for name, text in documents:
    doc_chunks = chunk_text(text, chunk_size=500, overlap=50)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([name] * len(doc_chunks))

print(f"Total chunks created: {len(all_chunks)}")
print("\nExample chunk:\n")
print(textwrap.fill(all_chunks[0], width=100))


## 6. Create embeddings for each chunk

We use `all-MiniLM-L6-v2`, a small, fast, high-quality sentence-embedding model from `sentence-transformers`.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # normalize so we can use cosine similarity via inner product
)

print("Embedding matrix shape:", chunk_embeddings.shape)


## 7. Build a FAISS vector index (the retrieval store)

In [ ]:
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # inner product == cosine similarity (since vectors are normalized)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")


## 8. Retrieval function

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "text": all_chunks[idx],
            "source": chunk_sources[idx],
            "score": float(score)
        })
    return results

# Quick sanity check
sample_query = "What is RAG?"
for r in retrieve(sample_query, top_k=3):
    print(f"[{r['source']} | score={r['score']:.3f}]")
    print(textwrap.fill(r['text'], width=100))
    print("-" * 100)


## 9. Generation model (LLM)

We use `google/flan-t5-base`, a free, instruction-tuned text-to-text model from Hugging Face,
so the whole pipeline runs locally without any API key. It's not as strong as GPT-4/Claude,
but it's great for demonstrating the RAG pattern end-to-end.

(See the optional section at the end of the notebook to swap in a hosted LLM API instead.)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Loading the model/tokenizer directly (instead of via pipeline(...)) avoids
# "Unknown task text2text-generation" errors that can happen with some
# transformers versions/environments where the pipeline task registry is incomplete.
gen_model_name = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)

def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([c["text"] for c in retrieved_chunks])
    prompt = f'''Answer the question using ONLY the context below. If the answer is not in the context, say "I don't know based on the given documents."

Context:
{context}

Question: {query}

Answer:'''
    return prompt

def generate_answer(query, retrieved_chunks):
    prompt = build_prompt(query, retrieved_chunks)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    output_ids = gen_model.generate(**inputs, max_new_tokens=200)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)


## 10. Full RAG pipeline: ask a question end-to-end

In [ ]:
def rag_answer(query, top_k=3, verbose=True):
    retrieved_chunks = retrieve(query, top_k=top_k)
    answer = generate_answer(query, retrieved_chunks)

    if verbose:
        print("QUESTION:", query)
        print("\nRETRIEVED CONTEXT:")
        for r in retrieved_chunks:
            print(f"  - ({r['source']}, score={r['score']:.3f}) {r['text'][:120]}...")
        print("\nANSWER:")
        print(textwrap.fill(answer, width=100))

    return answer

# Try it out
_ = rag_answer("What are the key components of a RAG system?")


In [ ]:
_ = rag_answer("Why is RAG useful for private or custom data?")


In [ ]:
# Try your own question here!
your_question = "What does the embedding model do in a RAG pipeline?"
_ = rag_answer(your_question)


## 11. (Optional) Use a hosted LLM instead of `flan-t5`

If you have access to a hosted LLM API (OpenAI, Anthropic, etc.) and want higher-quality answers,
you can replace the `generate_answer` function with a call to that API instead. Example using the
Anthropic API (requires `pip install anthropic` and an API key set as an environment variable):

```python
import anthropic

client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def generate_answer_anthropic(query, retrieved_chunks):
    prompt = build_prompt(query, retrieved_chunks)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# Then simply call:
# retrieved_chunks = retrieve("your question")
# generate_answer_anthropic("your question", retrieved_chunks)
```

## Next steps / ideas to extend this project
- Try larger/better generation models (e.g. `flan-t5-large`, `Mistral-7B-Instruct`, or a hosted API model).
- Add support for `.docx` files using `python-docx`.
- Store the FAISS index on disk (`faiss.write_index` / `faiss.read_index`) so you don't have to re-embed every run.
- Add a simple chat UI using Gradio or Streamlit.
- Add source citations to the final answer output.
- Try the Hugging Face datasets mentioned in the assignment brief instead of your own PDFs, for a bigger test set.
